# Model Checking DTMC with StormPy

In [32]:
import stormpy

path = "./engagement.pm"
prism_program = stormpy.parse_prism_program(path)
model = stormpy.build_model(prism_program)

print(model)
# --------------------------------------------------------------
# Model type: 	DTMC (sparse)
# States: 	5
# Transitions: 	12
# Reward Models:  none
# State Labels: 	9 labels
#    * deadlock -> 0 item(s)
#    * disengaged -> 1 item(s)
#    * success -> 1 item(s)
#    * converted -> 1 item(s)
#    * init -> 1 item(s)
#    * failure -> 1 item(s)
#    * engaged -> 1 item(s)
#    * browsing -> 1 item(s)
#    * abandoned -> 1 item(s)
# Choice Labels: 	none
# --------------------------------------------------------------

-------------------------------------------------------------- 
Model type: 	DTMC (sparse)
States: 	5
Transitions: 	12
Reward Models:  none
State Labels: 	9 labels
   * deadlock -> 0 item(s)
   * disengaged -> 1 item(s)
   * success -> 1 item(s)
   * converted -> 1 item(s)
   * init -> 1 item(s)
   * failure -> 1 item(s)
   * engaged -> 1 item(s)
   * browsing -> 1 item(s)
   * abandoned -> 1 item(s)
Choice Labels: 	none
-------------------------------------------------------------- 



## Reachability

In [9]:
# Compiling the reachability spec.
# The spec asks, "from a given state, what is the probability of eventually reaching converted?""
formula_str = """P=? [ F "converted" ]"""
properties = stormpy.parse_properties(formula_str, prism_program)

In [ ]:
# Build a model with labels and the state valuations.
options = stormpy.BuilderOptions([p.raw_formula for p in properties])
options.set_build_all_labels()
options.set_build_state_valuations()
model = stormpy.build_sparse_model_with_options(prism_program, options)

# Perform model checking for the reachability property.
result = stormpy.model_checking(model, properties[0])
assert result.result_for_all_states
values = result.get_values()

initial = set(model.initial_states)

print(f"{'idx':>3}  {'labels':<20}  {'P(F "converted")':>16}")
print("-" * 54)
for state in model.states:
    names = ", ".join(sorted(model.labeling.get_labels_of_state(state.id)))
    print(f"{state.id:>3}  {names:<20}  {values[state.id]:>16.4f}")

# idx  labels                P(F "converted")
# ------------------------------------------------------
#   0  browsing, init                  0.5184
#   1  engaged                         0.6720
#   2  disengaged                      0.2400
#   3  abandoned, failure              0.0000
#   4  converted, success              1.0000


idx  labels                P(F "converted")
------------------------------------------------------
  0  browsing, init                  0.5184
  1  engaged                         0.6720
  2  disengaged                      0.2400
  3  abandoned, failure              0.0000
  4  converted, success              1.0000


In [ ]:
# Bounded reachability property.
# It asks, "from a given state, what is the probability of reaching converted within 10 steps?"
formula_bounded = """P=? [ F<=10 "converted" ]"""
properties_bounded = stormpy.parse_properties(formula_bounded, prism_program)

In [13]:
# Build a model with labels and the state valuations.
options = stormpy.BuilderOptions([p.raw_formula for p in properties])
options.set_build_all_labels()
options.set_build_state_valuations()
model = stormpy.build_sparse_model_with_options(prism_program, options)

# Perform model checking for the reachability property.
result = stormpy.model_checking(model, properties[0])
assert result.result_for_all_states
values = result.get_values()

initial = set(model.initial_states)

print(f"{'idx':>3}  {'labels':<20}  {'P(F "converted")':>16}")
print("-" * 54)
for state in model.states:
    names = ", ".join(sorted(model.labeling.get_labels_of_state(state.id)))
    print(f"{state.id:>3}  {names:<20}  {values[state.id]:>16.4f}")

idx  labels                P(F "converted")
------------------------------------------------------
  0  browsing, init                  0.5184
  1  engaged                         0.6720
  2  disengaged                      0.2400
  3  abandoned, failure              0.0000
  4  converted, success              1.0000


## Invariance

In [ ]:
# Invariance. Probability of entering abandoned (failure state)
formula_invariance = """P=? [ G !"abandoned" ]"""
properties_invariance = stormpy.parse_properties(formula_invariance, prism_program)

In [ ]:
# Build a model with labels and the state valuations.
options = stormpy.BuilderOptions([p.raw_formula for p in properties_invariance])
options.set_build_all_labels()
options.set_build_state_valuations()
model = stormpy.build_sparse_model_with_options(prism_program, options)

# Perform model checking for the reachability property.
result = stormpy.model_checking(model, properties_invariance[0])
assert result.result_for_all_states

# Get the values for the invariance property.
values_invariance = result.get_values()

initial = set(model.initial_states)

print(f"{'idx':>3}  {'labels':<20}  {'P(G !"abandoned")':>16}")
print("-" * 54)
for state in model.states:
    names = ", ".join(sorted(model.labeling.get_labels_of_state(state.id)))
    print(f"{state.id:>3}  {names:<20}  {values_invariance[state.id]:>16.4f}")

# idx  labels                P(G !"abandoned")
# ------------------------------------------------------
#   0  browsing, init                  0.5184
#   1  engaged                         0.6720
#   2  disengaged                      0.2400
#   3  abandoned, failure              0.0000
#   4  converted, success              1.0000


idx  labels                P(G !"abandoned")
------------------------------------------------------
  0  browsing, init                  0.5184
  1  engaged                         0.6720
  2  disengaged                      0.2400
  3  abandoned, failure              0.0000
  4  converted, success              1.0000


In [ ]:
# Combining safety and reachability
# Tests the user not going to the failure state without being engaged.
formula_safety_reachability = """P=? [ !"abandoned" U "engaged" ]"""
properties_safety_reachability = stormpy.parse_properties(
    formula_safety_reachability, prism_program
)

In [ ]:
# Repeat the model building and checking process for the new property.
options = stormpy.BuilderOptions(
    [p.raw_formula for p in properties_safety_reachability]
)
options.set_build_all_labels()
options.set_build_state_valuations()
model = stormpy.build_sparse_model_with_options(prism_program, options)

# Perform model checking for the reachability property.
result = stormpy.model_checking(model, properties_safety_reachability[0])
assert result.result_for_all_states

# Get the values for the safety and reachability property.
values_safety_reachability = result.get_values()

initial = set(model.initial_states)

print(f"{'idx':>3}  {'labels':<20}  {'P(!"abandoned" U "engaged")':>16}")
print("-" * 54)
for state in model.states:
    names = ", ".join(sorted(model.labeling.get_labels_of_state(state.id)))
    print(f"{state.id:>3}  {names:<20}  {values_safety_reachability[state.id]:>16.4f}")

# idx  labels                P(!"abandoned" U "engaged")
# ------------------------------------------------------
#   0  browsing, init                  0.7714
#   1  engaged                         1.0000
#   2  disengaged                      0.3571
#   3  abandoned, failure              0.0000
#   4  converted, success              0.0000

idx  labels                P(!"abandoned" U "engaged")
------------------------------------------------------
  0  browsing, init                  0.7714
  1  engaged                         1.0000
  2  disengaged                      0.3571
  3  abandoned, failure              0.0000
  4  converted, success              0.0000


## Verification

In [31]:
# // Is conversion probability at least 50%?
verification_spec = """P>=0.5 [ F "converted" ]"""
prop = stormpy.parse_properties(verification_spec, prism_program)[0]


initial_state = next(iter(model.initial_states))
result = stormpy.model_checking(model, prop)
print(f"'Conversion probability is at least 50%': {result.at(initial_state)}")
# 'Conversion probability is at least 50%': True


'Conversion probability is at least 50%': True


# Model Checking MDP with StormPy

In [ ]:
import stormpy

path = "./activity_agent.pm"
prism_program = stormpy.parse_prism_program(path)
model = stormpy.build_model(prism_program)

print(model)
# --------------------------------------------------------------
# Model type: 	MDP (sparse)
# States: 	13
# Transitions: 	27
# Choices: 	15
# Reward Models:  task_completion
# State Labels: 	9 labels
#    * deadlock -> 0 item(s)
#    * s_success -> 1 item(s)
#    * s_abandon -> 4 item(s)
#    * s_W -> 1 item(s)
#    * s_WM -> 1 item(s)
#    * s_M -> 1 item(s)
#    * init -> 1 item(s)
#    * s_0 -> 1 item(s)
#    * done -> 4 item(s)
# Choice Labels: 	none
# --------------------------------------------------------------

-------------------------------------------------------------- 
Model type: 	MDP (sparse)
States: 	13
Transitions: 	27
Choices: 	15
Reward Models:  task_completion
State Labels: 	9 labels
   * deadlock -> 0 item(s)
   * s_success -> 1 item(s)
   * s_abandon -> 4 item(s)
   * s_W -> 1 item(s)
   * s_WM -> 1 item(s)
   * s_M -> 1 item(s)
   * init -> 1 item(s)
   * s_0 -> 1 item(s)
   * done -> 4 item(s)
Choice Labels: 	none
-------------------------------------------------------------- 



## Reachability

In [ ]:
# Compiling the reachability spec.
# The spec asks, "from a given state, what is the probability of eventually reaching converted?""
formula_pmax = """Pmax=? [ F "s_success" ]"""
formula_pmin = """Pmin=? [ F "s_success" ]"""

properties = {
    "Pmax": stormpy.parse_properties(formula_pmax, prism_program)[0],
    "Pmin": stormpy.parse_properties(formula_pmin, prism_program)[0],
}

# Build a model with labels and the state valuations.
options = stormpy.BuilderOptions([p.raw_formula for p in properties.values()])
options.set_build_all_labels()
options.set_build_state_valuations()
model = stormpy.build_sparse_model_with_options(prism_program, options)

initial_state = next(iter(model.initial_states))

# Perform model checking for the reachability property.
for name, prop in properties.items():
    result = stormpy.model_checking(model, prop)
    print(f"{name}: {result.at(initial_state):.4f}")
# Pmax: 0.6083
# Pmin: 0.5833

Pmax: 0.6083
Pmin: 0.5833


In [ ]:
# Compiling the bounded reachability spec.
formula_pmax = """Pmax=? [ F<=3 "s_success" ]"""
formula_pmin = """Pmin=? [ F<=3 "s_success" ]"""

properties = {
    "Pmax": stormpy.parse_properties(formula_pmax, prism_program)[0],
    "Pmin": stormpy.parse_properties(formula_pmin, prism_program)[0],
}

# Build a model with labels and the state valuations.
options = stormpy.BuilderOptions([p.raw_formula for p in properties.values()])
options.set_build_all_labels()
options.set_build_state_valuations()
model = stormpy.build_sparse_model_with_options(prism_program, options)

initial_state = next(iter(model.initial_states))

# Perform model checking for the reachability property.
for name, prop in properties.items():
    result = stormpy.model_checking(model, prop)
    print(f"{name}: {result.at(initial_state):.4f}")
# Pmax: 0.5850
# Pmin: 0.4200

Pmax: 0.5850
Pmin: 0.4200


## Invariance

In [ ]:
# Compiling the bounded reachability spec.
formula_inv_pmax = """Pmax=? [ G !"s_abandon" ]"""
formula_inv_pmin = """Pmin=? [ G !"s_abandon" ]"""
formula_combined_pmax = """Pmax=? [ !"s_abandon" U "s_success" ]"""
formula_combined_pmin = """Pmin=? [ !"s_abandon" U "s_success" ]"""

properties = {
    "Pmax Invariance": stormpy.parse_properties(formula_inv_pmax, prism_program)[0],
    "Pmin Invariance": stormpy.parse_properties(formula_inv_pmin, prism_program)[0],
    "Pmax Combined": stormpy.parse_properties(formula_combined_pmax, prism_program)[0],
    "Pmin Combined": stormpy.parse_properties(formula_combined_pmin, prism_program)[0],
}

# Build a model with labels and the state valuations.
options = stormpy.BuilderOptions([p.raw_formula for p in properties.values()])
options.set_build_all_labels()
options.set_build_state_valuations()
model = stormpy.build_sparse_model_with_options(prism_program, options)

initial_state = next(iter(model.initial_states))

# Perform model checking for the reachability property.
for name, prop in properties.items():
    result = stormpy.model_checking(model, prop)
    print(f"{name}: {result.at(initial_state):.4f}")
# Pmax Invariance: 0.6083
# Pmin Invariance: 0.5833
# Pmax Combined: 0.6083
# Pmin Combined: 0.5833

Pmax Invariance: 0.6083
Pmin Invariance: 0.5833
Pmax Combined: 0.6083
Pmin Combined: 0.5833


## Reward

In [ ]:
# Compiling the bounded reachability spec.
formula_reward_max = """Rmax=? [ F "done" ]"""
formula_reward_min = """Rmin=? [ F "done" ]"""

properties = {
    "Rmax": stormpy.parse_properties(formula_reward_max, prism_program)[0],
    "Rmin": stormpy.parse_properties(formula_reward_min, prism_program)[0]
}

# Build a model with labels and the state valuations.
options = stormpy.BuilderOptions([p.raw_formula for p in properties.values()])
options.set_build_all_labels()
options.set_build_state_valuations()
model = stormpy.build_sparse_model_with_options(prism_program, options)

initial_state = next(iter(model.initial_states))

# Perform model checking for the reachability property.
for name, prop in properties.items():
    result = stormpy.model_checking(model, prop)
    print(f"{name}: {result.at(initial_state):.4f}")
# Rmax: 6.0833
# Rmin: 5.8333

Rmax: 6.0833
Rmin: 5.8333
